# Spectra pooling, full run (Member 7)

**Question:** a test molecule has 1–16 spectra. Does a *learned* way of combining them (attention, DeepSets)
rank the true structure higher than averaging the per-spectrum predictions, and is the gain really from
pooling rather than from extra training?

**Built on Member 3's pipeline** (`baselines/b1_mass_only/casmi_pipeline.py`), so the numbers sit next to B1/B2:

| Piece | Source |
|---|---|
| Validation split (class 1 = structure known from other libraries, class 2 = new structure) | Member 3 `make_split`, salt `casmi-split-v1` (Member 5's shared split is not out yet) |
| Candidate pool | Member 3 `build_pool`: every training structure, 10 ppm window around the measured mass, no oracle |
| Per-spectrum model | Member 3 `ResidualMLP`, 2048-bit Morgan, trained here with `train_model` |
| Evaluation | Member 3 `evaluate`: MRR@25 by class, balanced = mean of classes 1 and 2, 95% bootstrap CI |

Only change to the split: queries keep up to **16** spectra per molecule (test has 1–16); the held-out
structures and their classes are identical to Member 3's.

**Methods, all on the same frozen per-spectrum model and the same candidates:**

| Method | What it is |
|---|---|
| `mean` | average per-spectrum probabilities (what B3/neural baselines do) |
| `mean_tuned` | **fair control**: the per-spectrum head gets the same extra training as the poolers, loss on the averaged probabilities |
| `deepsets` | arch2: rho(mean phi(h_i), log n) |
| `attention` | arch1: gated attention, learns a weight per spectrum |

Each learned method is trained with 3 seeds. Claims use paired bootstrap differences against `mean_tuned`.

**Outputs:** small results → `Project/outputs/pooling_v2/` (synced to the Mac). Heavy files (encoder, pooling
weights, artifact for the offline Kaggle notebook, caches) → `MyDrive/DATA266_CASMI/pooling_v2/`.

**Runtime:** about 3–4 h on an L4 (choose *L4 GPU*, High-RAM). Safe to leave overnight; the GPU is released at the end or on any error.

In [1]:
# ---- Config (env vars override; POOL_QUICK=1 is the local smoke test) ----
import os

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
E = os.environ.get
QUICK = E("POOL_QUICK", "0") == "1"

MAC = "/content/drive/Othercomputers/My Mac/data266-9067"
REPO = E("POOL_REPO", f"{MAC}/Project/01_repo/Team-10-Enveda-CASMI-2026")
RESULTS_DIR = E("POOL_RESULTS_DIR", f"{MAC}/Project/outputs/pooling_v2")           # small, synced
HEAVY_DIR = E("POOL_HEAVY_DIR", "/content/drive/MyDrive/DATA266_CASMI/pooling_v2")  # big, Drive
DRIVE_DATA = E("POOL_DRIVE_DATA", "/content/drive/MyDrive/DATA266_CASMI/data")     # train/test parquet
LOCAL_DATA = E("POOL_LOCAL_DATA", "/content/casmi_data")                           # fast local copy
CP_PATH = E("POOL_CP_PATH", f"{REPO}/baselines/b1_mass_only/casmi_pipeline.py")    # Member 3's pipeline

SEEDS = [0] if QUICK else [0, 1, 2]
N_VAL = int(E("POOL_N_VAL", 3000))           # validation molecules (Member 3 uses 3000)
MAX_QUERY_SPECTRA = 16                       # test molecules have 1-16 spectra
PER_STRUCTURE = int(E("POOL_PER_STRUCTURE", 16))   # training spectra kept per structure
PPM = 10.0
ENC_KW = {"n_bins": 1200, "width": 2048, "depth": 4, "dropout": 0.2}  # Member 3's ResidualMLP defaults
ENC_EPOCHS = int(E("POOL_ENC_EPOCHS", 20))
ENC_MAX_MIN = float(E("POOL_ENC_MAX_MIN", 150))    # time cap for the encoder
POOL_EPOCHS = int(E("POOL_POOL_EPOCHS", 20))
REUSE_ENCODER = E("POOL_REUSE_ENCODER", "1") == "1"  # skip stage A if HEAVY_DIR/encoder.pt exists
AUTO_DISCONNECT = E("POOL_AUTO_DISCONNECT", "1") == "1"
print("Colab:", IN_COLAB, "| quick:", QUICK, "| seeds:", SEEDS, "| encoder epochs:", ENC_EPOCHS)

Colab: True | quick: False | seeds: [0, 1, 2] | encoder epochs: 20


In [2]:
# ---- Setup: Drive, GPU release on error, imports ----
import importlib.util
import json
import shutil
import subprocess
import sys
import time
import urllib.request

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    def _release(shell, etype, evalue, tb, tb_offset=None):
        shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
        if AUTO_DISCONNECT:
            print("Error: releasing the GPU runtime.")
            from google.colab import runtime
            runtime.unassign()
    get_ipython().set_custom_exc((Exception,), _release)
try:
    import rdkit
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit==2026.3.3"], check=True)
    import rdkit
for p in (f"{REPO}/src", f"{REPO}/models/spectra_pooling"):
    if p not in sys.path:
        sys.path.insert(0, p)
for d in (RESULTS_DIR, HEAVY_DIR, LOCAL_DATA):
    os.makedirs(d, exist_ok=True)

# Member 3's pipeline: from the repo, or (until PR #5 is merged) from GitHub.
if not os.path.exists(CP_PATH):
    CP_PATH = f"{HEAVY_DIR}/casmi_pipeline.py"
    base = "https://raw.githubusercontent.com/ChaitanyaUppalapati/Team-10-Enveda-CASMI-2026"
    for ref in ("main", "member3_baseline"):
        try:
            urllib.request.urlretrieve(f"{base}/{ref}/baselines/b1_mass_only/casmi_pipeline.py", CP_PATH)
            print("casmi_pipeline.py from", ref)
            break
        except Exception as err:
            print(ref, "->", err)
spec = importlib.util.spec_from_file_location("casmi_pipeline", CP_PATH)
cp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cp)

import matplotlib
import numpy as np
import pandas as pd
import torch

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from common.full import (
    METHODS,
    batched_forward,
    embed_all,
    make_pool,
    paired_bootstrap,
    predict_molecules,
    rank_pooled,
    save_artifact,
    train_pool,
    unpack_bits,
)
from common.sets import group_indices, pad_sets

DEV = cp.get_device()
torch.manual_seed(0)
print("device:", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "",
      "| rdkit", rdkit.__version__, "| FP bits", cp.FP_BITS)
T0 = time.time()
def stamp(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

Mounted at /content/drive
main -> HTTP Error 404: Not Found
casmi_pipeline.py from member3_baseline
device: cuda NVIDIA L4 | rdkit 2026.03.3 | FP bits 2048


In [3]:
# ---- Data: train.parquet and test.parquet on Drive (Kaggle API once if missing), copied to local disk ----
def kaggle_fetch(name):
    from google.colab import userdata
    def secret(key):
        try:
            return userdata.get(key)
        except Exception:
            return None
    if secret("KAGGLE_API_TOKEN"):
        os.environ["KAGGLE_API_TOKEN"] = secret("KAGGLE_API_TOKEN")
    else:
        os.environ["KAGGLE_USERNAME"] = secret("KAGGLE_USERNAME") or ""
        os.environ["KAGGLE_KEY"] = secret("KAGGLE_KEY") or ""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle>=1.7.4"], check=True)
    dl = subprocess.run(["kaggle", "competitions", "download", "-c", "enveda-CASMI26-molecule-id-mass-spectra",
                         "-f", name, "-p", "/content/raw"], capture_output=True, text=True)
    print(dl.stdout[-1500:], dl.stderr[-1500:])
    if dl.returncode:
        raise RuntimeError("Kaggle download failed (message above). 401 = token; 403 = accept the rules.")
    for f in os.listdir("/content/raw"):
        if f.startswith(name) and f.endswith(".zip"):
            subprocess.run(["unzip", "-o", "-q", f"/content/raw/{f}", "-d", "/content/raw"], check=True)
    shutil.copy(f"/content/raw/{name}", f"{DRIVE_DATA}/{name}")

os.makedirs(DRIVE_DATA, exist_ok=True)
for name in ("train.parquet", "test.parquet"):
    if not os.path.exists(f"{DRIVE_DATA}/{name}"):
        kaggle_fetch(name)
    if not os.path.exists(f"{LOCAL_DATA}/{name}"):
        shutil.copy(f"{DRIVE_DATA}/{name}", f"{LOCAL_DATA}/{name}")
TRAIN_PATH = f"{LOCAL_DATA}/train.parquet"
stamp("data ready")


 
  0%|          | 0.00/4.62M [00:00<?, ?B/s]
 22%|██▏       | 1.00M/4.62M [00:01<00:04, 763kB/s]
 43%|████▎     | 2.00M/4.62M [00:01<00:01, 1.47MB/s]
 65%|██████▍   | 3.00M/4.62M [00:01<00:00, 2.28MB/s]
 87%|████████▋ | 4.00M/4.62M [00:01<00:00, 3.28MB/s]
100%|██████████| 4.62M/4.62M [00:01<00:00, 2.50MB/s]

[   1.1 min] data ready


In [4]:
# ---- Member 3's split (classes 1 and 2), candidate pool and fingerprints ----
meta = cp.load_meta(LOCAL_DATA)
queries, lib_mask = cp.make_split(meta, max_query_spectra=MAX_QUERY_SPECTRA)
queries = cp.sample_molecules(queries, N_VAL)
truth = cp.truth_table(queries).reset_index(drop=True)
assert not lib_mask[queries.row_id.to_numpy()].any(), "a query spectrum is still in the training library"
pool = cp.build_pool(meta, lib_mask)
fp_cache = f"{HEAVY_DIR}/pool_fp_{cp.FP_BITS}.npz"
if os.path.exists(fp_cache) and np.load(fp_cache, allow_pickle=True)["keys"].tolist() == pool.key.tolist():
    pool_fp = np.load(fp_cache, allow_pickle=True)["fp"]
else:
    pool_fp = cp.morgan_packed(pool.smiles.tolist())
    np.savez(fp_cache, fp=pool_fp, keys=np.array(pool.key.tolist(), dtype=object))
key_index = {k: i for i, k in enumerate(pool.key.to_numpy())}
n_spec = queries.groupby("molecule_id").size()
print(f"pool {len(pool):,} structures | val molecules {len(truth):,} "
      f"(class 1: {(truth.cls == 1).sum()}, class 2: {(truth.cls == 2).sum()}, "
      f"np proxy: {(truth.query_lib == 'enveda-np-examples').sum()}) | spectra per molecule: "
      f"median {n_spec.median():.0f}, max {n_spec.max()}")
stamp("split ready")

pool 275,810 structures | val molecules 2,912 (class 1: 1413, class 2: 1499, np proxy: 250) | spectra per molecule: median 4, max 16
[   3.4 min] split ready


In [5]:
# ---- Training spectra (structures outside the validation queries' libraries) and validation spectra ----
train_rows = cp.select_train_rows(meta, lib_mask, per_structure=PER_STRUCTURE)
train_rows = train_rows[train_rows.inchikey14.isin(key_index)].reset_index(drop=True)
tr_mz, tr_in = cp.read_spectra(TRAIN_PATH, train_rows.row_id.to_numpy())
va_mz, va_in = cp.read_spectra(TRAIN_PATH, queries.row_id.to_numpy())
data = {"mz": tr_mz, "intensity": tr_in, "precursor": train_rows.precursor_mz.to_numpy(np.float32),
        "adduct": cp.adduct_ids(train_rows.adduct.astype(str)),
        "target_index": train_rows.inchikey14.map(key_index).to_numpy(np.int64), "fp_packed": pool_fp}
val = {"pool": pool, "pool_fp": pool_fp, "queries": queries, "mz": va_mz, "intensity": va_in, "truth": truth}
print(f"{len(train_rows):,} training spectra from {train_rows.inchikey14.nunique():,} structures "
      f"| {len(queries):,} validation spectra")
stamp("spectra read")

1,661,739 training spectra from 247,666 structures | 14,242 validation spectra
[   4.9 min] spectra read


In [6]:
# ---- Stage A: per-spectrum model (Member 3's ResidualMLP), shared by every pooling method ----
models = cp.build_models()
enc = models["residual_mlp"](**ENC_KW)
enc_path = f"{HEAVY_DIR}/encoder.pt"
if REUSE_ENCODER and os.path.exists(enc_path) and not QUICK:
    enc.load_state_dict(torch.load(enc_path, map_location="cpu"))
    enc = enc.to(DEV).eval()
    enc_hist = pd.read_csv(f"{HEAVY_DIR}/encoder_history.csv")
    print("reused encoder from", enc_path)
else:
    enc, enc_hist = cp.train_model(enc, data, val, DEV, epochs=ENC_EPOCHS, max_minutes=ENC_MAX_MIN,
                                   out_path=enc_path)
    enc_hist.to_csv(f"{HEAVY_DIR}/encoder_history.csv", index=False)
enc = enc.to(DEV).eval()
for p in enc.parameters():
    p.requires_grad_(False)
display(enc_hist.tail(3))
stamp("stage A done")

{'epoch': 1, 'minutes': 1.22, 'train_bce': 0.08350999358989793, 'val_MRR@25': 0.4468, 'class1': 0.4861, 'class2': 0.4075, 'np_proxy': 0.4703}
{'epoch': 2, 'minutes': 2.45, 'train_bce': 0.0682775935333205, 'val_MRR@25': 0.569, 'class1': 0.5875, 'class2': 0.5504, 'np_proxy': 0.5946}
{'epoch': 3, 'minutes': 3.67, 'train_bce': 0.06493872596964079, 'val_MRR@25': 0.6146, 'class1': 0.6239, 'class2': 0.6054, 'np_proxy': 0.6425}
{'epoch': 4, 'minutes': 4.89, 'train_bce': 0.06249607362438608, 'val_MRR@25': 0.6566, 'class1': 0.6638, 'class2': 0.6495, 'np_proxy': 0.6886}
{'epoch': 5, 'minutes': 6.11, 'train_bce': 0.060517890354150616, 'val_MRR@25': 0.6811, 'class1': 0.6994, 'class2': 0.6628, 'np_proxy': 0.7097}
{'epoch': 6, 'minutes': 7.33, 'train_bce': 0.058849298274388485, 'val_MRR@25': 0.6928, 'class1': 0.7088, 'class2': 0.6768, 'np_proxy': 0.7236}
{'epoch': 7, 'minutes': 8.56, 'train_bce': 0.057391771219544495, 'val_MRR@25': 0.6988, 'class1': 0.7115, 'class2': 0.686, 'np_proxy': 0.7128}
{'epoc

,epoch,minutes,train_bce,val_MRR@25,class1,class2,np_proxy
17,18,22.12,0.048683,0.7501,0.7747,0.7255,0.7582
18,19,23.38,0.048503,0.7504,0.7737,0.7271,0.7577
19,20,24.62,0.048423,0.7503,0.7738,0.7267,0.7594


[  29.6 min] stage A done


In [7]:
# ---- Embeddings (frozen) and the sets each method sees ----
H_tr = embed_all(enc, tr_mz, tr_in, data["precursor"], data["adduct"], DEV)
H_va = embed_all(enc, va_mz, va_in, queries.precursor_mz.to_numpy(np.float32),
                 cp.adduct_ids(queries.adduct.astype(str)), DEV)
EMB = H_tr.shape[1]
groups = group_indices(train_rows.inchikey14.to_numpy().astype(str))
train_keys = sorted(groups)
target_fp = {k: pool_fp[key_index[k]] for k in train_keys}
qpos = queries.assign(pos=np.arange(len(queries)))
val_sets = [qpos.loc[qpos.molecule_id == m, "pos"].to_numpy() for m in truth.molecule_id]
val_bits = unpack_bits(torch.as_tensor(np.stack([pool_fp[key_index[k]] for k in truth.true_key])))
masses = qpos.groupby("molecule_id").neutral_mass.median().reindex(truth.molecule_id)
set_size = np.array([len(s) for s in val_sets])
del tr_mz, tr_in, data
print("embeddings: train", tuple(H_tr.shape), "val", tuple(H_va.shape))
stamp("embeddings done")

embeddings: train (1661739, 2048) val (14242, 2048)
[  30.0 min] embeddings done


In [8]:
# ---- Stage B: train mean_tuned (control), DeepSets and attention, 3 seeds each; evaluate everything ----
pool_keys = pool.key.to_numpy()
results, rr, histories, att_weights, states = [], {}, [], None, {}

def evaluate_probs(method, seed, probs):
    rankings = rank_pooled(cp, pool, pool_fp, masses, probs, ppm=PPM)
    table = cp.evaluate(rankings, truth, pool_keys, ci=True)
    rr[(method, seed)] = cp.rank_table(rankings, truth, pool_keys).rr.to_numpy()
    for group, row in table.iterrows():
        results.append({"method": method, "seed": seed, "group": group, **row.to_dict()})
    return table

with torch.no_grad():
    m_probs = []
    for s in range(0, len(val_sets), 1024):
        h, mask = pad_sets(val_sets[s:s + 1024], H_va)
        p = torch.sigmoid(enc.head(h.to(DEV).float()).float())
        m = mask.to(DEV).unsqueeze(-1).float()
        m_probs.append(((p * m).sum(1) / m.sum(1).clamp(min=1)).cpu())
print("mean (no training):")
display(evaluate_probs("mean", -1, torch.cat(m_probs).numpy()))

for seed in SEEDS:
    for method in ["mean_tuned", "deepsets", "attention"]:
        stamp(f"{method} seed {seed}")
        head = make_pool(method, EMB, cp.FP_BITS, head=enc.head)
        head, hist = train_pool(head, H_tr, groups, train_keys, target_fp, DEV, H_va, val_sets, val_bits,
                                epochs=POOL_EPOCHS, seed=seed, ckpt=f"{HEAVY_DIR}/pool_{method}_s{seed}.pt")
        histories.append(hist.assign(method=method, seed=seed))
        with torch.no_grad():
            logits, w = batched_forward(head, val_sets, H_va, DEV)
        table = evaluate_probs(method, seed, torch.sigmoid(logits).cpu().numpy())
        print(method, seed, "balanced MRR@25:", table.loc["balanced", "MRR@25"])
        states[(method, seed)] = {k: v.cpu() for k, v in head.state_dict().items()}
        if method == "attention" and seed == SEEDS[0]:
            att_weights = w
stamp("stage B done")

mean (no training):


,n,MRR@25,top1,top5,top25,MRR_ci_low,MRR_ci_high
class 1,1413.0,0.7740,0.6808,0.8931,0.9639,0.7570,0.7931
class 2,1499.0,0.7268,0.6191,0.8619,0.9533,0.7100,0.7443
np proxy,250.0,0.7577,0.6440,0.9040,0.9840,0.7132,0.8013
balanced,2912.0,0.7504,0.6500,0.8775,0.9586,0.7335,0.7687


[  30.0 min] mean_tuned seed 0
  epoch 1: train 0.0394  val 0.0502
  epoch 2: train 0.0385  val 0.0504
  epoch 3: train 0.0382  val 0.0504
  epoch 4: train 0.0379  val 0.0504
  epoch 5: train 0.0377  val 0.0503
  epoch 6: train 0.0375  val 0.0504
  epoch 7: train 0.0373  val 0.0503
  epoch 8: train 0.0371  val 0.0503
  epoch 9: train 0.0369  val 0.0503
  epoch 10: train 0.0367  val 0.0503
  epoch 11: train 0.0366  val 0.0503
  epoch 12: train 0.0364  val 0.0502
  epoch 13: train 0.0363  val 0.0502
  epoch 14: train 0.0361  val 0.0502
  epoch 15: train 0.0359  val 0.0502
  epoch 16: train 0.0359  val 0.0502
  epoch 17: train 0.0357  val 0.0502
  epoch 18: train 0.0357  val 0.0502
  epoch 19: train 0.0356  val 0.0502
  epoch 20: train 0.0356  val 0.0502
mean_tuned 0 balanced MRR@25: 0.7786
[  36.0 min] deepsets seed 0
  epoch 1: train 0.0704  val 0.0624
  epoch 2: train 0.0578  val 0.0599
  epoch 3: train 0.0557  val 0.0586
  epoch 4: train 0.0545  val 0.0579
  epoch 5: train 0.0537  val

In [9]:
# ---- Results: mean ± std over seeds, paired differences against the fair control ----
res = pd.DataFrame(results)
summary_tbl = (res.groupby(["group", "method"])["MRR@25"].agg(["mean", "std"]).round(4)
               .unstack("method").reindex(["balanced", "class 1", "class 2", "np proxy"]))
print("MRR@25 (mean over seeds; std in the second block)")
display(summary_tbl)

def per_molecule(method):
    keys = [k for k in rr if k[0] == method]
    return np.mean([rr[k] for k in keys], axis=0)   # average reciprocal rank over seeds

cls = truth.cls.to_numpy()
pairs = [("attention", "mean_tuned"), ("deepsets", "mean_tuned"), ("attention", "deepsets"),
         ("mean_tuned", "mean"), ("attention", "mean")]
diffs = []
for a, b in pairs:
    ra, rb = per_molecule(a), per_molecule(b)
    for group, sel in [("all", np.ones(len(cls), bool)), ("class 1", cls == 1), ("class 2", cls == 2),
                       ("2+ spectra", set_size >= 2)]:
        diffs.append({"comparison": f"{a} - {b}", "group": group, "n": int(sel.sum()),
                      **paired_bootstrap(ra[sel], rb[sel])})
diffs = pd.DataFrame(diffs).round(4)
diffs["significant"] = (diffs.ci_low > 0) | (diffs.ci_high < 0)
print("Paired differences in MRR@25 (95% bootstrap CI). 'all' averages both classes as they occur.")
display(diffs)

buckets = pd.cut(set_size, [0, 1, 3, 7, 16], labels=["1", "2-3", "4-7", "8-16"])
by_size = pd.DataFrame({m: pd.Series(per_molecule(m)).groupby(np.asarray(buckets)).mean()
                        for m in METHODS}).round(4)
by_size["n"] = pd.Series(np.asarray(buckets)).value_counts()
display(by_size)

MRR@25 (mean over seeds; std in the second block)


mean                                   std                \
method   attention deepsets    mean mean_tuned attention deepsets mean   
group                                                                    
balanced    0.7237   0.7246  0.7504     0.7776    0.0010   0.0017  NaN   
class 1     0.7522   0.7546  0.7740     0.8126    0.0016   0.0029  NaN   
class 2     0.6951   0.6947  0.7268     0.7427    0.0008   0.0007  NaN   
np proxy    0.7313   0.7252  0.7577     0.7726    0.0103   0.0068  NaN   

                     
method   mean_tuned  
group                
balanced     0.0009  
class 1      0.0006  
class 2      0.0014  
np proxy     0.0011

Paired differences in MRR@25 (95% bootstrap CI). 'all' averages both classes as they occur.


,comparison,group,n,delta,ci_low,ci_high,significant
0,attention - mean_tuned,all,2912,-0.0538,-0.0607,-0.0465,True
1,attention - mean_tuned,class 1,1413,-0.0603,-0.0710,-0.0501,True
2,attention - mean_tuned,class 2,1499,-0.0476,-0.0569,-0.0384,True
3,attention - mean_tuned,2+ spectra,2492,-0.0518,-0.0598,-0.0443,True
4,deepsets - mean_tuned,all,2912,-0.0528,-0.0601,-0.0456,True
5,deepsets - mean_tuned,class 1,1413,-0.0579,-0.0686,-0.0473,True
6,deepsets - mean_tuned,class 2,1499,-0.0480,-0.0578,-0.0385,True
7,deepsets - mean_tuned,2+ spectra,2492,-0.0509,-0.0586,-0.0434,True
8,attention - deepsets,all,2912,-0.0009,-0.0046,0.0027,False
9,attention - deepsets,class 1,1413,-0.0024,-0.0079,0.0031,False


,mean,mean_tuned,deepsets,attention,n
1,0.6852,0.7154,0.6514,0.6499,420
2-3,0.7326,0.7540,0.7022,0.7016,735
4-7,0.7633,0.7891,0.7378,0.7396,1172
8-16,0.7903,0.8238,0.7747,0.7682,585


In [10]:
# ---- Does attention favour the better spectra? (molecules with >= 2 spectra, first seed) ----
with torch.no_grad():
    single = torch.sigmoid(enc.head(H_va.to(DEV).float()).float()).cpu().numpy()
bits = val_bits.numpy()
picked, average = [], []
for s, w, y in zip(val_sets, att_weights, bits, strict=True):
    if len(s) < 2:
        continue
    p = single[s]
    q = (p * y).sum(1) / np.maximum(p.sum(1) + y.sum() - (p * y).sum(1), 1e-8)   # quality of each spectrum alone
    picked.append(q[np.argmax(w)])
    average.append(q.mean())
picked, average = np.array(picked), np.array(average)
att_check = {"molecules": int(len(picked)), "top_weighted_spectrum_quality": float(picked.mean()),
             "average_spectrum_quality": float(average.mean()),
             "share_top_weighted_beats_average": float(np.mean(picked > average))}
print(json.dumps(att_check, indent=2))

{
  "molecules": 2492,
  "top_weighted_spectrum_quality": 0.37315046787261963,
  "average_spectrum_quality": 0.31736570596694946,
  "share_top_weighted_beats_average": 0.7969502407704655
}


In [11]:
# ---- Best method -> artifact for the offline Kaggle notebook, and a submission.csv for test.parquet ----
bal = res[(res.group == "balanced") & (res.method != "mean")].groupby("method")["MRR@25"].mean()
best_method = bal.idxmax()
best_seed = res[(res.group == "balanced") & (res.method == best_method)].set_index("seed")["MRR@25"].idxmax()
art_dir = f"{HEAVY_DIR}/artifact"
save_artifact(art_dir, ENC_KW, {k: v.cpu() for k, v in enc.state_dict().items()}, best_method,
              states[(best_method, best_seed)], EMB, cp.FP_BITS,
              {"ppm": PPM, "split": "casmi-split-v1", "seed": int(best_seed)})
pool[["key", "smiles", "mass", "n_ref"]].to_parquet(f"{art_dir}/pool.parquet")
np.save(f"{art_dir}/pool_fp.npy", pool_fp)
shutil.copy(CP_PATH, f"{art_dir}/casmi_pipeline.py")

test, test_path = cp.load_test(LOCAL_DATA)
te_mz, te_in = cp.read_spectra(test_path, test.row_id.to_numpy())
best_pool = make_pool(best_method, EMB, cp.FP_BITS, head=enc.head)
best_pool.load_state_dict(states[(best_method, best_seed)])
t_masses, t_probs = predict_molecules(cp, enc, best_pool.to(DEV).eval(), test, te_mz, te_in, DEV)
t_rankings = rank_pooled(cp, pool, pool_fp, t_masses, t_probs, ppm=PPM)
sub = cp.write_submission(t_rankings, pool, test, path=f"{RESULTS_DIR}/submission_{best_method}.csv")
print(f"best: {best_method} (seed {best_seed}) | artifact -> {art_dir} | submission rows: {len(sub)}")

best: mean_tuned (seed 0) | artifact -> /content/drive/MyDrive/DATA266_CASMI/pooling_v2/artifact | submission rows: 400


In [12]:
# ---- Save small results to the synced folder ----
res.to_csv(f"{RESULTS_DIR}/results_by_seed.csv", index=False)
summary_tbl.to_csv(f"{RESULTS_DIR}/mrr_summary.csv")
diffs.to_csv(f"{RESULTS_DIR}/paired_differences.csv", index=False)
by_size.to_csv(f"{RESULTS_DIR}/mrr_by_set_size.csv")
pd.concat(histories).to_csv(f"{RESULTS_DIR}/pooling_history.csv", index=False)
enc_hist.to_csv(f"{RESULTS_DIR}/encoder_history.csv", index=False)
mean_tbl = res.groupby(["method", "group"])["MRR@25"].mean().unstack()
log_rows = [{"date": time.strftime("%Y-%m-%d"), "version": f"M7-{m}", "change": f"{m} pooling on B3-style ResidualMLP",
             "val_MRR@25": float(mean_tbl.loc[m, "balanced"]), "class1": float(mean_tbl.loc[m, "class 1"]),
             "class2": float(mean_tbl.loc[m, "class 2"]), "np_proxy": float(mean_tbl.loc[m, "np proxy"]), "LB": ""}
            for m in METHODS]
pd.DataFrame(log_rows).to_csv(f"{RESULTS_DIR}/experiment_log_rows.csv", index=False)
summary = {"date": time.strftime("%Y-%m-%d %H:%M"), "runtime_min": round((time.time() - T0) / 60, 1),
           "device": str(DEV), "gpu": torch.cuda.get_device_name(0) if DEV.type == "cuda" else "",
           "quick": QUICK, "seeds": SEEDS, "n_val": int(len(truth)), "n_train_spectra": int(len(train_rows)),
           "pool_size": int(len(pool)), "config": {"enc": ENC_KW, "enc_epochs": ENC_EPOCHS,
           "pool_epochs": POOL_EPOCHS, "per_structure": PER_STRUCTURE, "ppm": PPM, "fp_bits": cp.FP_BITS},
           "balanced_mrr": {m: float(mean_tbl.loc[m, "balanced"]) for m in METHODS},
           "class2_mrr": {m: float(mean_tbl.loc[m, "class 2"]) for m in METHODS},
           "best": {"method": best_method, "seed": int(best_seed)}, "attention_check": att_check,
           "split": "Member 3 make_split casmi-split-v1, up to 16 spectra per query"}
with open(f"{RESULTS_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
plot = res[res.group.isin(["class 1", "class 2", "balanced"])].groupby(["group", "method"])["MRR@25"]
plot.mean().unstack().reindex(columns=METHODS).plot.bar(ax=axes[0], rot=0,
                                                        yerr=plot.std().unstack().reindex(columns=METHODS))
axes[0].set_ylabel("MRR@25")
axes[0].set_title("Pooling method by class (mean ± std over seeds)")
by_size[METHODS].plot.bar(ax=axes[1], rot=0)
axes[1].set_xlabel("spectra per molecule")
axes[1].set_title("By number of spectra")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/pooling_results.png", dpi=150)
plt.close()
print(json.dumps(summary["balanced_mrr"], indent=2))
print("saved to", RESULTS_DIR)

{
  "mean": 0.7504,
  "mean_tuned": 0.7776333333333333,
  "deepsets": 0.7246333333333334,
  "attention": 0.7237
}
saved to /content/drive/Othercomputers/My Mac/data266-9067/Project/outputs/pooling_v2


In [13]:
# ---- Done: release the GPU ----
stamp("all done")
if IN_COLAB and AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()

[  75.8 min] all done
